# 02 — Data Cleaning & Target Preservation
---
**Purpose:** Implement the exact logic from the approved data cleaning specification on the complete 101k lap dataset. Creates diagnostic flags and a final `is_clean` definition. Computes extensive data loss reporting by season, circuit, team, and compound.

**Outputs:** 
- `outputs/cleaned_laps.parquet` (Full dataset with diagnostic flags)
- `outputs/clean_laps.parquet` (Subset where `is_clean == True`)
- `outputs/cleaning_report.csv` (Reporting metrics)

In [1]:
import pandas as pd
import numpy as np
import os, warnings
warnings.filterwarnings("ignore")

OUTPUT_DIR = os.path.join("..", "outputs")
input_path = os.path.join(OUTPUT_DIR, "combined_laps.parquet")

print("Loading dataset...")
df = pd.read_parquet(input_path)
total_raw_laps = len(df)
print(f"Loaded {total_raw_laps:,} laps across {df['RaceId'].nunique()} races.")

Loading dataset...
Loaded 101,290 laps across 92 races.


## 1. Delete Forbidden / Redundant Columns

In [2]:
# Drop string timings since we have _Seconds floats
string_cols = [
    "Time", "LapTime", "PitInTime", "PitOutTime", 
    "Sector1Time", "Sector2Time", "Sector3Time",
    "Sector1SessionTime", "Sector2SessionTime", "Sector3SessionTime",
    "LapStartTime"
]
# Drop target leakage columns (Gaps represent the END of the lap)
leakage_cols = [
    "GapToLeader", "IntervalToPositionAhead",
    "GapToLeaderSeconds", "IntervalToPositionAheadSeconds"
]

to_drop = [c for c in string_cols + leakage_cols if c in df.columns]
df = df.drop(columns=to_drop)
print(f"Dropped {len(to_drop)} redundant/leakage columns.")

Dropped 15 redundant/leakage columns.


## 2. Generate Diagnostic Flags

In [3]:
# Using the specific flag names requested where appropriate, and mapping to our logic.

# 1. Missing target
df['flag_missing_target'] = df['LapTimeSeconds'].isna() | (df['LapTimeSeconds'] <= 0)

# 2. Missing crucial tyre data
df['flag_missing_tyre'] = df['Compound'].isna() | df['TyreLife'].isna() | df['Stint'].isna()

# 3. Pit lane interference
df['is_pit_in'] = df['PitInTimeSeconds'].notna()
df['is_pit_out'] = df['PitOutTimeSeconds'].notna()

# 4. Standing starts
df['flag_lap_1'] = df['LapNumber'] == 1

# 5. Non-Dry tyres
df['flag_wet_compound'] = df['Compound'].isin(['INTERMEDIATE', 'WET'])

# 6. Race Control (SC, VSC, Red Flag)
df['is_safety_car'] = df['HasSafetyCar'] == True
df['is_vsc'] = (df['HasVSC'] == True) | (df['HasVSCEnding'] == True)
df['is_red_flag'] = df['HasRedFlag'] == True

# 7. Local Yellow (without green clearing it in the same lap)
df['is_yellow_flag'] = df['HasYellow'] & ~df['HasGreen']

# 8. Unreliable Telemetry/Sensors
df['is_deleted'] = df['Deleted'] == True
df['is_accurate'] = df['IsAccurate'].fillna(True)

# 9. Light Rain on Dry Tyres
df['flag_rainfall'] = df['Rainfall'] == True

# 10. Extreme Outliers (115% of median race/compound pace)
median_pace = df.groupby(['RaceId', 'Compound'])['LapTimeSeconds'].transform('median')
df['flag_outlier'] = (df['LapTimeSeconds'] > (median_pace * 1.15)) & df['LapTimeSeconds'].notna()

# Collect all exclusionary flags (Note: is_accurate is positive, so we invert it for exclusion)
exclusion_flags = [
    'flag_missing_target', 'flag_missing_tyre', 'is_pit_in', 'is_pit_out',
    'flag_lap_1', 'flag_wet_compound', 'is_safety_car', 'is_vsc', 'is_red_flag',
    'is_yellow_flag', 'is_deleted', 'flag_rainfall', 'flag_outlier'
]

# Create inverted accurate flag for the sum
df['flag_inaccurate'] = ~df['is_accurate']
exclusion_flags.append('flag_inaccurate')

print("Flags generated successfully.")

Flags generated successfully.


## 3. Data Loss Analysis & Reporting

In [4]:
# Overall loss by flag
report_data = []
print(f"{'Flag Reason':<25} {'Laps Excluded':>15} {'% of Total':>12}")
print("-" * 55)
for flag in exclusion_flags:
    count = df[flag].sum()
    pct = (count / total_raw_laps) * 100
    report_data.append({'Metric': flag, 'Category': 'Overall', 'Laps_Excluded': count, 'Percentage': pct})
    print(f"{flag:<25} {count:>15,} {pct:>11.1f}%")

df['is_clean_lap'] = ~df[exclusion_flags].any(axis=1)
clean_count = df['is_clean_lap'].sum()
overall_retention = (clean_count / total_raw_laps) * 100
report_data.append({'Metric': 'Total_Clean', 'Category': 'Overall', 'Laps_Excluded': total_raw_laps - clean_count, 'Percentage': overall_retention})

print(f"\nTotal Raw Laps:   {total_raw_laps:,}")
print(f"Total Clean Laps: {clean_count:,}")
print(f"Clean Retention:  {overall_retention:.1f}%")

Flag Reason                 Laps Excluded   % of Total
-------------------------------------------------------
flag_missing_target                 1,539         1.5%
flag_missing_tyre                   1,055         1.0%
is_pit_in                           3,469         3.4%
is_pit_out                          3,467         3.4%
flag_lap_1                          1,826         1.8%
flag_wet_compound                   6,406         6.3%
is_safety_car                       5,365         5.3%
is_vsc                              2,012         2.0%
is_red_flag                           224         0.2%
is_yellow_flag                        516         0.5%
is_deleted                          1,273         1.3%
flag_rainfall                       4,011         4.0%
flag_outlier                        7,520         7.4%
flag_inaccurate                    13,768        13.6%

Total Raw Laps:   101,290
Total Clean Laps: 79,865
Clean Retention:  78.8%


## 4. Dimensional Reporting

In [5]:
def analyze_dimension(dim_col):
    res = df.groupby(dim_col).agg(
        Total_Laps=('LapNumber', 'count'),
        Clean_Laps=('is_clean_lap', 'sum')
    )
    res['Retention_Pct'] = (res['Clean_Laps'] / res['Total_Laps']) * 100
    res['Loss_Pct'] = 100 - res['Retention_Pct']
    return res

dimensions = ['Year', 'GrandPrix', 'CanonicalTeam', 'Compound']

for dim in dimensions:
    if dim in df.columns:
        res = analyze_dimension(dim)
        print(f"\n--- Retention by {dim} ---")
        print(res[['Total_Laps', 'Clean_Laps', 'Retention_Pct']].sort_values('Retention_Pct'))
        
        # Add to report
        for idx, row in res.iterrows():
            report_data.append({
                'Metric': str(idx),
                'Category': dim,
                'Laps_Excluded': row['Total_Laps'] - row['Clean_Laps'],
                'Percentage': row['Loss_Pct']
            })

report_df = pd.DataFrame(report_data)
report_out = os.path.join(OUTPUT_DIR, "cleaning_report.csv")
report_df.to_csv(report_out, index=False)
print(f"\nSaved extensive report to {report_out}")


--- Retention by Year ---
      Total_Laps  Clean_Laps  Retention_Pct
Year                                       
2022       23577       17225      73.058489
2025       26689       21339      79.954288
2024       26604       21280      79.987972
2023       24420       20021      81.986077

--- Retention by GrandPrix ---
                           Total_Laps  Clean_Laps  Retention_Pct
GrandPrix                                                       
British Grand Prix               3571        2093      58.611033
Australian Grand Prix            3973        2473      62.245155
São Paulo Grand Prix             4752        2961      62.310606
Canadian Grand Prix              5202        3600      69.204152
Monaco Grand Prix                5356        3856      71.994025
Japanese Grand Prix              3353        2448      73.009245
Dutch Grand Prix                 5525        4210      76.199095
Belgian Grand Prix               3328        2541      76.352163
Singapore Grand Prix       


Saved extensive report to ..\outputs\cleaning_report.csv


## 5. Save Datasets

In [6]:
# 1. Full processed dataset (with flags)
cleaned_out = os.path.join(OUTPUT_DIR, "cleaned_laps.parquet")
df.to_parquet(cleaned_out, index=False)
print(f"Saved complete processed dataset to: {cleaned_out} ({(os.path.getsize(cleaned_out)/1e6):.1f} MB)")

# 2. Subset for tyre modelling (only clean laps)
clean_subset_out = os.path.join(OUTPUT_DIR, "clean_laps.parquet")
clean_df = df[df['is_clean_lap']].copy()
# Drop flags since they are all False in this subset to save space
cols_to_drop = exclusion_flags + ['is_clean_lap', 'is_accurate']
clean_df = clean_df.drop(columns=[c for c in cols_to_drop if c in clean_df.columns])
clean_df.to_parquet(clean_subset_out, index=False)
print(f"Saved clean-lap subset to: {clean_subset_out} ({(os.path.getsize(clean_subset_out)/1e6):.1f} MB)")

print("\n[OK] Notebook 02 Data Cleaning complete.")

Saved complete processed dataset to: ..\outputs\cleaned_laps.parquet (6.6 MB)


Saved clean-lap subset to: ..\outputs\clean_laps.parquet (5.0 MB)

[OK] Notebook 02 Data Cleaning complete.
